# 03 — Loss Landscape and Optimization Dynamics in Physics-Informed Neural Networks

> *"Optimization is the art of choosing the right surface to descend."* — folklore

Notebooks 1 and 2 established that a PINN is a smooth parametric function $u_\theta$ and that its derivatives — including those defining the PDE residual — are computable exactly by automatic differentiation. We now turn to the central practical question: **why is training a PINN so hard, and what mathematical structure governs success or failure?**

We will see that:

* The composite loss carries internally conflicting components whose gradients can be wildly mismatched in scale and direction;
* Neural networks exhibit a *spectral bias* that prefers low frequencies, exactly the wrong inductive bias for many PDEs;
* The neural tangent kernel (NTK) gives a quantitative theory of these effects;
* The community has converged on a *two-stage* optimizer (Adam → L-BFGS) for principled reasons rooted in stochastic optimization vs. quasi-Newton theory;
* Several adaptive weighting and sampling strategies provide partial cures.

Throughout we keep the analysis at the level of mathematical statistics, calculus of variations, and convex/non-convex optimization theory.

## Table of Contents

1. Anatomy of the composite loss $\mathcal{L}_{\text{PINN}}$
2. Collocation points as Monte Carlo quadrature
3. Sampling strategies: uniform, Sobol, Latin Hypercube, RAR, RAD
4. Gradient pathologies — competing scales and directions
5. The neural tangent kernel (NTK) of a PINN
6. Spectral bias and the failure to capture high-frequency content
7. Curing pathologies: loss-weight adaptation, normalization, causality
8. First-order stochastic optimization — Adam
9. Quasi-Newton optimization — L-BFGS and the curvature of the loss
10. The two-stage strategy — Adam → L-BFGS, justified
11. Diagnostics and convergence criteria

## 1. Anatomy of the Composite Loss

Recall (Notebook 1) the PINN loss

$$
\mathcal{L}_{\text{PINN}}(\theta) \;=\; \lambda_r \mathcal{L}_r(\theta) + \lambda_b \mathcal{L}_b(\theta) + \lambda_0 \mathcal{L}_0(\theta) + \lambda_d \mathcal{L}_d(\theta),
$$

with

$$
\mathcal{L}_r = \frac{1}{N_r}\sum_{i=1}^{N_r}\big|\mathcal{N}[u_\theta](x^r_i,t^r_i) - f(x^r_i,t^r_i)\big|^2 \;\;\text{(PDE residual)},
$$
$$
\mathcal{L}_b = \frac{1}{N_b}\sum_i\big|\mathcal{B}[u_\theta](x^b_i,t^b_i) - g(x^b_i,t^b_i)\big|^2 \;\;\text{(boundary)},
$$
$$
\mathcal{L}_0 = \frac{1}{N_0}\sum_i\big|u_\theta(x^0_i,0) - u_0(x^0_i)\big|^2 \;\;\text{(initial)},
$$
$$
\mathcal{L}_d = \frac{1}{N_d}\sum_i\big|u_\theta(x^d_i,t^d_i) - u^d_i\big|^2 \;\;\text{(data)}.
$$

### 1.1 What each term encodes

* $\mathcal{L}_r$ — the *physics* itself: the PDE in the bulk. Computing it costs nested AD.
* $\mathcal{L}_b, \mathcal{L}_0$ — the *boundary/initial value problem*. These pin the solution: a homogeneous PDE has infinitely many solutions; the data of BCs/ICs selects one.
* $\mathcal{L}_d$ — *measurement / supervision*. In the inverse problem this is the dominant source of identifiability for unknown parameters.

### 1.2 The loss is a Monte Carlo estimator

Each empirical sum is an unbiased estimator of an integral over the domain:

$$
\mathcal{L}_r(\theta) \;\approx\; \frac{1}{|\Omega_T|}\int_{\Omega\times(0,T]}\!\big|\mathcal{N}[u_\theta] - f\big|^2\,dx\,dt,
$$

where $\Omega_T = \Omega\times(0,T]$. Variance scales as $\sigma^2 / N_r$. Hence increasing $N_r$ is *the* knob for noise reduction — but at linear cost. Quasi-Monte Carlo sequences give variance $\mathcal{O}((\log N)^d / N^2)$ in low dimensions and are universally recommended for PINN sampling.

### 1.3 Non-convexity of the pullback

Even when the continuum functional $J[u]$ is *convex* in $u$ (as in linear elliptic PDEs), the pullback $\theta \mapsto J(u_\theta)$ is **non-convex** because $\theta\mapsto u_\theta$ is non-affine (compositions of $\tanh$ and linear maps). The non-convexity of deep learning thus inherits into PINNs in full force. Saddle points abound; *bad* local minima exist for many PDEs (Krishnapriyan et al. 2021).

## 2. Collocation Points — Quadrature Without a Mesh

Collocation points play the role that quadrature nodes play in FEM, *but they are not tied to a triangulation*. They are simply the empirical support on which the PDE is enforced.

### 2.1 Population vs. empirical risk

Population (continuum) residual functional:

$$
\mathcal{R}(\theta) \;:=\; \mathbb{E}_{(x,t)\sim\mu}\!\big[\big|\mathcal{N}[u_\theta](x,t) - f(x,t)\big|^2\big].
$$

Empirical residual:

$$
\widehat{\mathcal{R}}_{N_r}(\theta) \;=\; \frac{1}{N_r}\sum_{i=1}^{N_r}\big|\mathcal{N}[u_\theta](x^r_i,t^r_i) - f(x^r_i,t^r_i)\big|^2.
$$

By the strong law of large numbers $\widehat{\mathcal{R}}_{N_r}\to \mathcal{R}$ pointwise in $\theta$; uniform convergence over a compact $\theta$-region follows from Rademacher complexity bounds on the parametric family.

### 2.2 The generalization gap

The PINN *generalization error* (Mishra–Molinaro 2021) decomposes

$$
\underbrace{\|u_\theta - u^\star\|_{H^m}}_{\text{total}} \;\le\; \underbrace{C_1\,\widehat{\mathcal{R}}_{N_r}(\theta)^{1/2}}_{\text{empirical}} + \underbrace{C_2\,\mathfrak{R}_{N_r}(\mathcal{F})}_{\text{Rademacher}},
$$

with $\mathfrak{R}_{N_r}$ the Rademacher complexity of the residual class. *Driving training loss to zero is necessary but not sufficient* — we also need enough samples relative to the network's effective capacity.

## 3. Sampling Strategies

### 3.1 Uniform i.i.d. — the baseline

Samples drawn $\mathrm{Unif}(\Omega\times(0,T])$. Simple, unbiased, but high variance and uneven coverage in any finite draw.

### 3.2 Quasi-Monte Carlo: Sobol, Halton, Latin Hypercube

Low-discrepancy sequences fill the domain more uniformly. The Koksma–Hlawka inequality bounds the integration error by the discrepancy of the sequence times the variation of the integrand. For PINNs in low to moderate dimensions ($d \le 8$) QMC reduces variance by 1–2 orders of magnitude over uniform.

### 3.3 Residual-based adaptive sampling (RAR)

Wu et al. 2023: maintain a *pool* of candidate points, and at each refinement step add to the active set those with largest empirical residual. This concentrates samples where the PDE is hardest — the analog of adaptive mesh refinement (AMR).

### 3.4 Residual-based adaptive distribution (RAD)

Lu et al. 2021: define a probability density on $\Omega\times(0,T]$ proportional to $|r_\theta|^\alpha$ and resample from it periodically. With $\alpha \in [1,2]$, this drives mass toward steep-gradient regions (e.g. shock layers in Burgers) without depopulating smooth regions.

### 3.5 Causality-respecting sampling

For time-dependent PDEs, Wang–Sankaran–Perdikaris (2022) argue that *uniform* sampling in $t$ implicitly asks the network to fit the late-time solution before early-time errors are corrected — a violation of physical causality that destabilizes training. They propose weighting samples by $w(t) = \exp(-\epsilon\int_0^t \mathcal{L}_r(\tau)d\tau)$, effectively forcing the optimizer to satisfy early times before late times. This dramatically improves training for stiff time-dependent PDEs.

## 4. Gradient Pathologies — The Core Optimization Disease

Wang–Teng–Perdikaris (2021) diagnosed and named the primary failure mode of vanilla PINNs.

### 4.1 The mismatch problem

Each loss component induces a gradient $g_k(\theta) := \nabla_\theta \mathcal{L}_k(\theta)$. Vanilla SGD descends in the direction $-\sum_k \lambda_k g_k$. If the $g_k$ differ by orders of magnitude in norm, the dominant term hijacks descent — *one* loss reduces, the others stagnate.

Empirically, for stiff PDEs the residual gradient $g_r$ can be $10^2$ to $10^6$ times larger than the boundary gradient $g_b$. The network drives the PDE residual to zero by violating the boundary conditions — settling into a *non-physical* fixed point of $\mathcal{L}_{\text{PINN}}$.

### 4.2 Diagnostic: gradient histograms

Plotting $\{|g_r(\theta)|, |g_b(\theta)|, |g_0(\theta)|\}$ as histograms during training reveals the scale mismatch. The standard sign of pathology: $\max(|g_r|) \gg \text{mean}(|g_b|)$.

### 4.3 Adaptive weighting (Learning Rate Annealing, Wang et al. 2021)

Update $\lambda_k$ on the fly to equalize the *mean magnitudes*:

$$
\widehat\lambda_k \;=\; \frac{\max_\theta|\nabla_\theta \mathcal{L}_r(\theta)|}{\overline{|\nabla_\theta \mathcal{L}_k(\theta)|}},\qquad \lambda_k \leftarrow (1-\alpha)\lambda_k + \alpha \widehat\lambda_k.
$$

This is a heuristic but principled: it amounts to *preconditioning* the multi-objective gradient by the inverse Jacobian of the loss vector.

### 4.4 NTK-based weighting

Section 5 develops a deeper theory in which the right weights are determined by the *eigenvalues of the NTK restricted to each loss subspace*.

### 4.5 Hard constraints — eliminating $\mathcal{L}_b, \mathcal{L}_0$ altogether

Construct the ansatz so that BCs and ICs are satisfied by construction:

$$
u_\theta(x,t) \;=\; A(x,t) + D(x,t)\,\tilde u_\theta(x,t),
$$

where $A$ matches the BCs/ICs and $D$ is a smooth distance function vanishing on the appropriate parts of $\partial(\Omega\times[0,T])$. Then $u_\theta$ exactly satisfies the constraints regardless of $\tilde u_\theta$, removing the source of the gradient mismatch. Used widely with great success.

## 5. The Neural Tangent Kernel of a PINN

The NTK (Jacot–Gabriel–Hongler 2018) provides a quantitative theory of training dynamics in the infinite-width limit. For PINNs the NTK is itself a composite object that explains both the gradient pathology and the spectral bias.

### 5.1 The NTK in the standard setting

Let $\theta_t$ evolve by gradient flow $\dot\theta = -\nabla_\theta \mathcal{L}(\theta)$. For a scalar regression loss $\mathcal{L} = \tfrac{1}{2}\sum_i (u_\theta(x_i) - y_i)^2$, the evolution of predictions is

$$
\dot u_\theta(x_i) \;=\; \sum_j K_\theta(x_i,x_j)\,(y_j - u_\theta(x_j)),
$$

where $K_\theta(x,x') := \nabla_\theta u_\theta(x)\cdot \nabla_\theta u_\theta(x')$ is the **neural tangent kernel**. In the infinite-width limit (Jacot et al.), $K_\theta$ becomes deterministic and *constant* during training: training reduces to kernel regression with $K$.

### 5.2 The PINN NTK

Wang–Wang–Perdikaris (2022) extended this to PINNs. Decompose the loss into *blocks* (residual, boundary, initial, data). Each block induces an NTK matrix; for the residual block,

$$
K_{rr}(x,x') := \big\langle \nabla_\theta\,\mathcal{N}[u_\theta](x), \nabla_\theta\,\mathcal{N}[u_\theta](x')\big\rangle.
$$

The full PINN NTK is a block matrix

$$
\mathbf{K} \;=\; \begin{pmatrix} K_{rr} & K_{rb} & K_{r0} \\ K_{br} & K_{bb} & K_{b0} \\ K_{0r} & K_{0b} & K_{00}\end{pmatrix},
$$

and gradient flow on $\mathcal{L}_{\text{PINN}}$ produces convergence rates governed by the *eigenvalues of $\mathbf{K}$ restricted to the corresponding subspace*.

### 5.3 Spectral imbalance

Crucially, applying a differential operator $\mathcal{N}$ to $u_\theta$ *amplifies* high-frequency content of the NTK by powers of frequency. For a second-order operator the NTK eigenvalue at wavenumber $\xi$ scales like $|\xi|^4 \cdot \lambda^{NTK}_u(\xi)$. This produces an extreme condition-number disparity between blocks: $K_{rr}$ is dominated by high frequencies while $K_{bb}$ is broadband — exactly the gradient pathology, now seen as a *spectral phenomenon*.

### 5.4 NTK-guided weights

Wang et al. propose $\lambda_k \propto \operatorname{tr}(\mathbf{K})/\operatorname{tr}(K_{kk})$ — weighting each block inversely to its trace. This roughly equalizes the contraction rates across blocks and dramatically stabilizes training.

## 6. Spectral Bias of Neural Networks

### 6.1 The phenomenon

Rahaman et al. (2019) and Basri et al. (2019) showed that fully-connected networks with standard initializations and smooth activations learn the *low-frequency* components of their target functions exponentially faster than high-frequency components. The convergence rate at wavenumber $\xi$ is approximately

$$
\|u_\theta^{(t)} - u^\star\|_\xi \;\sim\; \exp(-\lambda(\xi)\,t),\qquad \lambda(\xi) \;\propto\; |\xi|^{-\alpha},
$$

with $\alpha > 0$ a function of architecture. The kernel of a wide network is heavily concentrated in the low-frequency subspace.

### 6.2 Why this is fatal for many PDEs

Many physically interesting solutions are *not* low-frequency: shock fronts (Burgers), boundary layers (Navier–Stokes), sharp solitons (KdV), oscillatory wave solutions (Helmholtz at high $k$). A vanilla PINN spends almost all its training budget fitting the smooth bulk and *fails to resolve the sharp features* — the very features that motivate the problem.

### 6.3 Mitigation

* **Fourier feature embedding** (Tancik et al. 2020): replace inputs $(x,t)$ by $[\cos(2\pi Bx),\sin(2\pi Bx)]$ with $B \sim \mathcal{N}(0,\sigma^2)$. By choosing $\sigma$ to match the target frequencies, the NTK becomes nearly stationary and broadband — spectral bias is suppressed.
* **SIREN** (Sitzmann et al. 2020): use $\sin(\omega_0 z)$ activations with $\omega_0 \sim 30$, broadening the NTK directly.
* **Multi-scale architectures (M-FFN, M-PINN)**: ensembles or multi-headed networks trained on different frequency bands.

### 6.4 A unifying perspective

Spectral bias and gradient pathology are two faces of the same coin: the NTK $\mathbf{K}$ of a PINN has a *poorly conditioned* spectrum, with eigenvectors clustered around low-frequency, low-block-weight modes. All practical PINN improvements — adaptive weights, NTK weighting, Fourier features, hard constraints, causal sampling — work by *flattening* the spectrum of $\mathbf{K}$.

## 7. Curing the Pathologies — A Summary Table

| Symptom | Diagnostic | Remedy | Where it lives |
|---|---|---|---|
| BC violations dominate | $|g_r| \gg |g_b|$ | Loss weighting (LR annealing, NTK weighting, GradNorm) | Loss design |
| Sharp features unresolved | Plateau in $\mathcal{L}_r$, smooth $u_\theta$ | Fourier features, SIREN, multi-scale nets | Architecture |
| Late-time error overwrites early-time | $\mathcal{L}_r(t)$ peaks at small $t$ | Causal sampling/weighting | Sampling |
| Stiff PDE, narrow boundary layer | Slow convergence overall | Adaptive collocation (RAR/RAD), AMR-like resampling | Sampling |
| Bad local minimum, oscillatory output | $\mathcal{L}$ stagnates with high variance | Switch to L-BFGS or restart from new init | Optimizer |
| Loss never decreases below $10^{-2}$ | Even L-BFGS plateaus | Domain decomposition (XPINN, Notebook 4) | Method |

## 8. First-Order Stochastic Optimization — Adam

### 8.1 The role of Adam

Adam (Kingma–Ba 2015) maintains exponential moving averages of gradients and squared gradients:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t,\quad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2,\quad \hat m_t = \frac{m_t}{1-\beta_1^t},\quad \hat v_t = \frac{v_t}{1-\beta_2^t},
$$
$$
\theta_{t+1} = \theta_t - \eta \frac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}.
$$

The per-coordinate adaptive step size $\eta/\sqrt{\hat v_t}$ acts as a diagonal preconditioner — coordinates with persistently large gradients receive smaller updates, those with small gradients are accelerated. This is an *implicit* form of scale-correction that partially mitigates the gradient-pathology problem.

### 8.2 Why Adam first?

* **Robust to bad initial geometry.** The early loss landscape is dominated by sharp ridges; second-order methods amplify noise here.
* **Stochastic-friendly.** Mini-batches of collocation points keep memory bounded; Adam's momentum cancels noise across iterations.
* **Wide basin of attraction.** Adam routinely finds a region near a good minimum even from random initialization, where curvature methods can later refine.

### 8.3 Limitations

Adam's $\sqrt{v_t}$ preconditioner is diagonal — it cannot rotate gradient directions. For ill-conditioned loss surfaces (which PINNs invariably produce) it stagnates at residual losses around $10^{-3}$ to $10^{-5}$, far from the $10^{-8}$–$10^{-12}$ accuracy needed for scientific applications. This is precisely where L-BFGS earns its place.

## 9. Quasi-Newton Optimization — L-BFGS

### 9.1 Newton's method and its impossibility for deep networks

Newton's method updates $\theta_{t+1} = \theta_t - H(\theta_t)^{-1}\nabla\mathcal{L}(\theta_t)$ where $H$ is the Hessian. Convergence is *quadratic* near a strict minimum. But $H$ is $|\theta|\times|\theta|$ — for a million-parameter PINN, neither storable nor invertible.

### 9.2 Quasi-Newton approximations

BFGS maintains a rank-2 update of an approximation $B_t \approx H(\theta_t)$ using only gradient and step information:

$$
B_{t+1} = B_t + \frac{y_t y_t^\top}{y_t^\top s_t} - \frac{B_t s_t s_t^\top B_t}{s_t^\top B_t s_t},
$$

with $s_t = \theta_{t+1}-\theta_t$, $y_t = g_{t+1}-g_t$. The inverse $H_t = B_t^{-1}$ satisfies the *secant equation* $H_{t+1} y_t = s_t$.

### 9.3 L-BFGS — the limited-memory variant

Liu–Nocedal (1989) store only the last $m$ pairs $(s_k,y_k)$ (typically $m\in[5,20]$) and compute $H_t g_t$ implicitly via the *two-loop recursion*, with cost $\mathcal{O}(mn)$ per step where $n=|\theta|$. Memory and compute are linear in $n$ — usable for million-parameter networks.

### 9.4 Why L-BFGS thrives on PINN losses

* **Curvature awareness.** The implicit Hessian approximation captures the highly anisotropic curvature of the PINN loss surface, performing *rotations* of gradient directions invisible to Adam's diagonal preconditioner.
* **Line search.** Wolfe-condition line search guarantees sufficient decrease per step, eliminating step-size guessing.
* **Full-batch operation.** L-BFGS requires deterministic gradients; in PINNs this is natural because collocation samples are typically resampled at low frequency or held fixed.

### 9.5 Failure modes

L-BFGS *only* works near a good minimum. From random initialization it tends to settle in a nearby saddle or low-quality basin. It also struggles with non-deterministic gradients (mini-batch sampling) because the secant equation assumes smoothness.

## 10. The Two-Stage Strategy — Adam → L-BFGS

The canonical PINN training pipeline, established by Raissi et al. (2019) and reaffirmed by virtually every successful paper since, is:

1. **Stage A (exploration).** Adam for $K_A \in [10^4, 10^5]$ iterations with full-batch or large mini-batches of collocation points. Goal: enter a wide basin of low loss; the final residual is typically $10^{-3}$ to $10^{-5}$.
2. **Stage B (refinement).** L-BFGS for $K_B \in [10^3, 10^4]$ iterations with all collocation points fixed. Goal: drive the loss down by 3–5 additional orders of magnitude using curvature information.

### 10.1 Optimization-theoretic justification

* **Stochastic vs. deterministic regimes.** Far from a minimum, stochastic methods enjoy *implicit regularization*: noise smooths the loss and lets the iterate cross saddle points. Near a minimum, deterministic methods enjoy *fast local convergence*. Adam exploits the first regime; L-BFGS the second.
* **First-order vs. second-order convergence.** Adam: at best $\mathcal{O}(1/t)$ in convex regimes, slower in non-convex. L-BFGS near a minimum: superlinear (Dennis–Moré conditions). The combined schedule is *the* fastest-known practical pipeline for PINN-scale problems.
* **Curvature mismatch.** PINN loss surfaces have a *power-law* spectrum: a few directions with curvature $10^4$, others with curvature $10^{-4}$. First-order methods are slow along low-curvature directions; second-order methods recover direction information needed to make progress there.

### 10.2 Hyperparameter conventions

* Adam learning rate $\eta = 10^{-3}$ initially with exponential decay (rate $\sim 0.95$ per $10^4$ steps).
* L-BFGS history $m = 10$ to $20$, max iters $5\cdot 10^3$, gradient tolerance $10^{-8}$, Wolfe line search with $c_1=10^{-4}$, $c_2=0.9$.
* Switch criterion: Adam loss has plateaued below $10^{-3}$ or after a fixed budget.

### 10.3 Variants and successors

* **NNCG / second-order trust regions** (Wang–Karniadakis 2024). Use a Gauss–Newton approximation of the residual Hessian: $H \approx J^\top J$ with $J = \nabla_\theta(\mathcal{N}[u_\theta] - f)$. Cheaper than full Hessian, more curvature than BFGS.
* **Natural gradient.** Preconditioning by the Fisher information matrix; equivalent to a particular Riemannian metric on $\Theta$.
* **Mixed-precision and bf16 fine-tuning.** Not common in PINNs because of precision requirements (Notebook 2).

## 11. Diagnostics and Convergence Criteria

A trained PINN should pass a battery of checks before being trusted as a PDE solver.

### 11.1 Per-component loss curves

Plot $\mathcal{L}_r, \mathcal{L}_b, \mathcal{L}_0, \mathcal{L}_d$ separately on a log scale. All terms should decrease together; if one stagnates orders of magnitude above the others, gradient pathology persists.

### 11.2 Residual field visualization

Sample a dense grid (just for visualization) and plot $|r_\theta(x,t)|$. Hot spots reveal regions where the PDE is poorly satisfied — typically shocks, boundary layers, or initial-condition mismatches. RAR/RAD sampling should target exactly these regions.

### 11.3 Comparison against high-fidelity reference

Where available, compare $u_\theta$ to a Chebyshev pseudospectral or fine FEM solution. Report $\|u_\theta - u^\star\|_{L^2}$ and the relative $L^\infty$ error. For Burgers (Notebook 5) the reference is from a high-order pseudospectral solver.

### 11.4 Physical sanity checks

* Conservation laws (mass, momentum, energy) where exact in the PDE.
* Maximum principles (for elliptic and parabolic PDEs).
* Symmetry of the solution under PDE symmetries.

### 11.5 Stopping criteria

* Adam stage: stop when $\mathcal{L}$ has not improved by more than $0.1\%$ over $10^3$ iterations.
* L-BFGS stage: stop when $\|\nabla\mathcal{L}\|_\infty < 10^{-8}$ or step size drops below $10^{-12}$.
* Failure: $\mathcal{L}$ plateaus above $10^{-3}$ after L-BFGS — restart with new initialization, new sampling, or domain decomposition.

## 12. Synthesis

PINN training is a study in *coupled* mathematical objects:

* A non-convex empirical risk on a smooth parametric submanifold of a Sobolev space;
* A composite loss whose components are PDE residuals and constraint penalties of inherently different scale;
* A neural tangent kernel $\mathbf{K}$ whose spectrum, deformed by the differential operator $\mathcal{N}$, directly governs convergence rate per frequency;
* A spectral bias in neural networks that biases the iteration toward low-frequency content;
* A two-stage optimizer that pairs the global exploration of Adam with the local quadratic refinement of L-BFGS.

The community's most successful PINN improvements — adaptive weighting, Fourier feature embedding, hard-constraint output transforms, causal sampling, residual-adaptive collocation — all act on the spectrum of $\mathbf{K}$, equalizing the convergence rates across components and frequencies. Notebook 4 takes this further: by *structurally* re-deriving the PINN ansatz from conservation principles, weak forms, or domain decomposition, we recover finer control of the same spectrum, at the cost of architectural complexity.

## References

1. **Wang, S., Teng, Y., Perdikaris, P.** *Understanding and mitigating gradient flow pathologies in physics-informed neural networks.* SIAM J. Sci. Comput. 43 (2021).
2. **Wang, S., Wang, H., Perdikaris, P.** *When and why PINNs fail to train: A neural tangent kernel perspective.* J. Comp. Phys. 449 (2022).
3. **Jacot, A., Gabriel, F., Hongler, C.** *Neural tangent kernel: Convergence and generalization in neural networks.* NeurIPS 2018.
4. **Rahaman, N. et al.** *On the spectral bias of neural networks.* ICML 2019.
5. **Tancik, M. et al.** *Fourier features let networks learn high frequency functions in low dimensional domains.* NeurIPS 2020.
6. **Wang, S., Sankaran, S., Perdikaris, P.** *Respecting causality is all you need for training physics-informed neural networks.* arXiv 2203.07404.
7. **Wu, C., Zhu, M., Tan, Q., Kartha, Y., Lu, L.** *A comprehensive study of non-adaptive and residual-based adaptive sampling for PINNs.* CMAME 403 (2023).
8. **Krishnapriyan, A. et al.** *Characterizing possible failure modes in physics-informed neural networks.* NeurIPS 2021.
9. **Kingma, D., Ba, J.** *Adam: A method for stochastic optimization.* ICLR 2015.
10. **Liu, D. C., Nocedal, J.** *On the limited memory BFGS method for large scale optimization.* Math. Programming 45 (1989).
11. **Nocedal, J., Wright, S. J.** *Numerical Optimization.* 2nd ed., Springer, 2006.
12. **Mishra, S., Molinaro, R.** *Estimates on the generalization error of physics-informed neural networks.* IMA J. Numer. Anal. 42 (2022).